## Primary models
- `Qwen/Qwen2.5-7B-Instruct`
- `mistralai/Mistral-7B-Instruct-v0.3`
- `meta-llama/Llama-3.1-8B-Instruct`

## Exploratory models
- `Qwen/Qwen2.5-3B-Instruct`
- `meta-llama/Llama-3.2-3B-Instruct`
- `humain-ai/ALLaM-7B-Instruct-preview`

## Conditions
- `EN`
- `EN_MK`
- `MK`
- `MK_EN`

The same Belebele-style prompt is used for every model and condition. Candidate answers are scored by conditional log-likelihood.


In [1]:
#!pip -q install -U transformers accelerate bitsandbytes huggingface_hub sentencepiece pandas numpy tqdm scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is 

In [1]:
%pip -q install bitsandbytes==0.50.2

In [3]:
import os
import gc
import shutil
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab: Runtime > Change runtime type > GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1),
    "GB",
)


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB


## 1. Mount Google Drive and load the dataset

In [4]:
from google.colab import drive, files

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/mk_en_csw_evaluation")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_DIR / "belebele_csw_final_895_generated.csv"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Upload belebele_csw_final_895_generated.csv")
    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError("No file uploaded.")

    uploaded_name = next(iter(uploaded))
    shutil.copy2(Path("/content") / uploaded_name, DATA_PATH)

print("Dataset:", DATA_PATH)
print("Results:", RESULTS_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset: /content/drive/MyDrive/mk_en_csw_evaluation/belebele_csw_final_895_generated.csv
Results: /content/drive/MyDrive/mk_en_csw_evaluation/results


In [13]:
df = pd.read_csv(DATA_PATH)

REQUIRED_COLUMNS = [
    "id", "link", "question_number",
    "en_passage", "en_question", "en_A", "en_B", "en_C", "en_D", "en_correct",
    "mk_passage", "mk_question", "mk_A", "mk_B", "mk_C", "mk_D", "mk_correct",
    "en_mk_passage", "en_mk_question",
    "mk_en_passage", "mk_en_question",
    "en_mk_switch_count", "mk_en_switch_count",
    "en_mk_switch_ratio", "mk_en_switch_ratio",
]

missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
assert not missing, f"Missing columns: {missing}"
assert len(df) == 895
assert df["id"].is_unique

DEV_IDS = {86, 458, 525, 614, 691}
assert DEV_IDS.isdisjoint(set(df["id"].astype(int)))

assert (
    df["en_correct"].astype(int).values
    == df["mk_correct"].astype(int).values
).all()

print("Dataset integrity checks: PASS")
print("Rows:", len(df))


Dataset integrity checks: PASS
Rows: 895


## 2. Hugging Face authentication

In [ ]:
from huggingface_hub import login, whoami

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print(whoami())
else:
    print("HF_TOKEN not found. Public models can still be used; gated Llama models cannot.")

## 3. Model and experiment configuration

In [17]:
PRIMARY_MODEL_SPECS = {
    "qwen2.5-7b": {
        "model_id": "Qwen/Qwen2.5-7B-Instruct",
        "revision": None,
        "group": "primary",
    },
    "mistral-7b": {
        "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "revision": None,
        "group": "primary",
    },
    "llama3.1-8b": {
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "revision": None,
        "group": "primary",
    },
}

EXPLORATORY_MODEL_SPECS = {
    "qwen2.5-3b": {
        "model_id": "Qwen/Qwen2.5-3B-Instruct",
        "revision": None,
        "group": "exploratory",
    },
    "llama3.2-3b": {
        "model_id": "meta-llama/Llama-3.2-3B-Instruct",
        "revision": None,
        "group": "exploratory",
    },
    "allam-7b": {
        "model_id": "humain-ai/ALLaM-7B-Instruct-preview",
        "revision": "a895c826737f20b4e1caedbf5fbf6f1cb6c88b8a",
        "group": "exploratory",
    },
}

MODEL_SPECS = {
    **PRIMARY_MODEL_SPECS,
    **EXPLORATORY_MODEL_SPECS,
}

QUANTIZATION = "8bit"
BATCH_SIZE = 8

LETTERS = ["A", "B", "C", "D"]
CANDIDATE_TEXTS = [" A", " B", " C", " D"]

CONDITIONS = {
    "EN": {
        "passage": "en_passage",
        "question": "en_question",
        "answers": ["en_A", "en_B", "en_C", "en_D"],
        "correct": "en_correct",
        "switch_count": None,
        "switch_ratio": None,
    },
    "EN_MK": {
        "passage": "en_mk_passage",
        "question": "en_mk_question",
        "answers": ["en_A", "en_B", "en_C", "en_D"],
        "correct": "en_correct",
        "switch_count": "en_mk_switch_count",
        "switch_ratio": "en_mk_switch_ratio",
    },
    "MK": {
        "passage": "mk_passage",
        "question": "mk_question",
        "answers": ["mk_A", "mk_B", "mk_C", "mk_D"],
        "correct": "mk_correct",
        "switch_count": None,
        "switch_ratio": None,
    },
    "MK_EN": {
        "passage": "mk_en_passage",
        "question": "mk_en_question",
        "answers": ["mk_A", "mk_B", "mk_C", "mk_D"],
        "correct": "mk_correct",
        "switch_count": "mk_en_switch_count",
        "switch_ratio": "mk_en_switch_ratio",
    },
}

print("Primary models:")
for k, v in PRIMARY_MODEL_SPECS.items():
    print(k, "->", v["model_id"])

print("\nExploratory models:")
for k, v in EXPLORATORY_MODEL_SPECS.items():
    print(k, "->", v["model_id"], "revision=", v["revision"])


Primary models:
qwen2.5-7b -> Qwen/Qwen2.5-7B-Instruct
mistral-7b -> mistralai/Mistral-7B-Instruct-v0.3
llama3.1-8b -> meta-llama/Llama-3.1-8B-Instruct

Exploratory models:
qwen2.5-3b -> Qwen/Qwen2.5-3B-Instruct revision= None
llama3.2-3b -> meta-llama/Llama-3.2-3B-Instruct revision= None
allam-7b -> humain-ai/ALLaM-7B-Instruct-preview revision= a895c826737f20b4e1caedbf5fbf6f1cb6c88b8a


## 4. Optional exploratory-model access/tokenizer test

In [18]:
from transformers import AutoTokenizer

for key, spec in EXPLORATORY_MODEL_SPECS.items():
    print("=" * 90)
    print(key, spec["model_id"], "revision=", spec["revision"])

    try:
        tok = AutoTokenizer.from_pretrained(
            spec["model_id"],
            revision=spec["revision"],
            use_fast=True,
        )

        print("Access: OK")
        for text in CANDIDATE_TEXTS:
            print(
                repr(text),
                "->",
                tok.encode(text, add_special_tokens=False),
            )

    except Exception as e:
        print("FAILED:", type(e).__name__, e)


qwen2.5-3b Qwen/Qwen2.5-3B-Instruct revision= None
Access: OK
' A' -> [362]
' B' -> [425]
' C' -> [356]
' D' -> [422]
llama3.2-3b meta-llama/Llama-3.2-3B-Instruct revision= None
Access: OK
' A' -> [362]
' B' -> [426]
' C' -> [356]
' D' -> [423]
allam-7b humain-ai/ALLaM-7B-Instruct-preview revision= a895c826737f20b4e1caedbf5fbf6f1cb6c88b8a


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.23MB            

tokenizer.model: downloading bytes:           |  0.00B            

Access: OK
' A' -> [395]
' B' -> [482]
' C' -> [415]
' D' -> [526]


## 5. Prompt construction

In [19]:
def build_prompt(row, condition):
    cfg = CONDITIONS[condition]

    passage = str(row[cfg["passage"]])
    question = str(row[cfg["question"]]).strip()
    a, b, c, d = [str(row[col]) for col in cfg["answers"]]

    return (
        f"P: {passage}\n"
        f"Q: {question}\n"
        f"A: {a}\n"
        f"B: {b}\n"
        f"C: {c}\n"
        f"D: {d}\n"
        f"Answer:"
    )


def gold_letter(row, condition):
    correct_num = int(row[CONDITIONS[condition]["correct"]])
    assert correct_num in {1, 2, 3, 4}
    return LETTERS[correct_num - 1]


## 6. Model loading

In [20]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


def make_quantization_config(mode):
    if mode == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)

    if mode == "4bit":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

    if mode == "none":
        return None

    raise ValueError(f"Unknown quantization mode: {mode}")


def get_input_device(model):
    for param in model.parameters():
        if param.device.type != "meta":
            return param.device
    raise RuntimeError("Could not determine model input device.")


def load_model_and_tokenizer(model_key):
    spec = MODEL_SPECS[model_key]
    model_id = spec["model_id"]
    revision = spec.get("revision")

    print("Loading:", model_id)
    if revision:
        print("Revision:", revision)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        revision=revision,
        use_fast=True,
    )

    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is None:
            raise RuntimeError("Tokenizer has neither pad token nor EOS token.")
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    continuation_token_ids = [
        tokenizer.encode(text, add_special_tokens=False)
        for text in CANDIDATE_TEXTS
    ]

    print("Candidate tokenization:")
    for text, ids in zip(CANDIDATE_TEXTS, continuation_token_ids):
        print(repr(text), "->", ids)

    all_single_token = all(len(ids) == 1 for ids in continuation_token_ids)
    scoring_mode = (
        "single_token_fast"
        if all_single_token
        else "multi_token_generic"
    )

    print("Scoring mode:", scoring_mode)

    kwargs = {
        "device_map": "auto",
        "low_cpu_mem_usage": True,
        "dtype": torch.float16,
    }

    if revision is not None:
        kwargs["revision"] = revision

    quant_cfg = make_quantization_config(QUANTIZATION)

    if quant_cfg is not None:
        kwargs["quantization_config"] = quant_cfg

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        **kwargs,
    )

    model.eval()

    forward_params = inspect.signature(model.forward).parameters

    if "logits_to_keep" in forward_params:
        last_logit_kwarg = {"logits_to_keep": 1}
    elif "num_logits_to_keep" in forward_params:
        last_logit_kwarg = {"num_logits_to_keep": 1}
    else:
        last_logit_kwarg = {}

    return {
        "model": model,
        "tokenizer": tokenizer,
        "continuation_token_ids": continuation_token_ids,
        "scoring_mode": scoring_mode,
        "last_logit_kwarg": last_logit_kwarg,
        "model_id": model_id,
        "revision": revision,
        "group": spec["group"],
    }


## 7. Candidate scoring

In [21]:
@torch.inference_mode()
def score_single_token_batch(prompts, loaded):
    model = loaded["model"]
    tokenizer = loaded["tokenizer"]

    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False,
        add_special_tokens=True,
    )

    device = get_input_device(model)
    encoded = {k: v.to(device) for k, v in encoded.items()}

    outputs = model(
        **encoded,
        use_cache=False,
        **loaded["last_logit_kwarg"],
    )

    last_logits = outputs.logits[:, -1, :].float()

    candidate_ids = torch.tensor(
        [ids[0] for ids in loaded["continuation_token_ids"]],
        dtype=torch.long,
        device=last_logits.device,
    )

    selected_logits = last_logits.index_select(-1, candidate_ids)
    log_normalizer = torch.logsumexp(
        last_logits,
        dim=-1,
        keepdim=True,
    )

    return (selected_logits - log_normalizer).cpu().numpy()


@torch.inference_mode()
def score_one_continuation_generic(prompt, continuation_ids, loaded):
    model = loaded["model"]
    tokenizer = loaded["tokenizer"]

    prompt_ids = tokenizer.encode(
        prompt,
        add_special_tokens=True,
    )

    full_ids = prompt_ids + list(continuation_ids)

    input_ids = torch.tensor(
        [full_ids],
        dtype=torch.long,
        device=get_input_device(model),
    )

    attention_mask = torch.ones_like(input_ids)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )

    logits = outputs.logits[0].float()

    start = len(prompt_ids)
    total_logprob = 0.0

    for i in range(start, len(full_ids)):
        target_id = full_ids[i]
        token_logprob = torch.log_softmax(
            logits[i - 1],
            dim=-1,
        )[target_id]

        total_logprob += float(token_logprob.item())

    return total_logprob


def score_prompt_batch(prompts, loaded):
    if loaded["scoring_mode"] == "single_token_fast":
        return score_single_token_batch(prompts, loaded)

    output = []

    for prompt in prompts:
        row_scores = []

        for continuation_ids in loaded["continuation_token_ids"]:
            row_scores.append(
                score_one_continuation_generic(
                    prompt,
                    continuation_ids,
                    loaded,
                )
            )

        output.append(row_scores)

    return np.asarray(output, dtype=np.float64)


## 8. Resumable evaluation

In [11]:
def atomic_save_csv(frame, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def prediction_path_for(model_key, suffix=""):
    return RESULTS_DIR / f"{model_key}_{QUANTIZATION}{suffix}_predictions.csv"


def evaluate_model(model_key, limit_per_condition=None, suffix=""):
    output_path = prediction_path_for(model_key, suffix)

    if output_path.exists():
        results = pd.read_csv(output_path)
        print("Resuming from:", output_path)
        print("Existing predictions:", len(results))
    else:
        results = pd.DataFrame()

    done = set()

    if len(results):
        done = set(
            zip(
                results["id"].astype(int),
                results["condition"].astype(str),
            )
        )

    loaded = load_model_and_tokenizer(model_key)

    try:
        for condition in CONDITIONS:
            pending_rows = []

            for _, row in df.iterrows():
                key = (int(row["id"]), condition)

                if key not in done:
                    pending_rows.append(row)

            if limit_per_condition is not None:
                pending_rows = pending_rows[:limit_per_condition]

            print("\n" + "=" * 100)
            print(model_key, "|", condition)
            print("Pending:", len(pending_rows))

            if not pending_rows:
                continue

            for start in tqdm(range(0, len(pending_rows), BATCH_SIZE)):
                batch_rows = pending_rows[start:start + BATCH_SIZE]

                prompts = [
                    build_prompt(row, condition)
                    for row in batch_rows
                ]

                scores = score_prompt_batch(prompts, loaded)

                records = []

                for row, score_vec in zip(batch_rows, scores):
                    pred_idx = int(np.argmax(score_vec))
                    pred = LETTERS[pred_idx]
                    gold = gold_letter(row, condition)

                    sorted_scores = np.sort(score_vec)
                    top1_margin = float(
                        sorted_scores[-1] - sorted_scores[-2]
                    )

                    cfg = CONDITIONS[condition]

                    switch_count = (
                        0
                        if cfg["switch_count"] is None
                        else int(row[cfg["switch_count"]])
                    )

                    switch_ratio = (
                        0.0
                        if cfg["switch_ratio"] is None
                        else float(row[cfg["switch_ratio"]])
                    )

                    records.append({
                        "model_key": model_key,
                        "model_id": loaded["model_id"],
                        "model_revision": loaded["revision"],
                        "model_group": loaded["group"],
                        "quantization": QUANTIZATION,
                        "scoring_mode": loaded["scoring_mode"],
                        "id": int(row["id"]),
                        "link": row["link"],
                        "question_number": int(row["question_number"]),
                        "condition": condition,
                        "gold": gold,
                        "prediction": pred,
                        "correct": int(pred == gold),
                        "logprob_A": float(score_vec[0]),
                        "logprob_B": float(score_vec[1]),
                        "logprob_C": float(score_vec[2]),
                        "logprob_D": float(score_vec[3]),
                        "top1_margin": top1_margin,
                        "switch_count": switch_count,
                        "switch_ratio": switch_ratio,
                    })

                results = pd.concat(
                    [results, pd.DataFrame(records)],
                    ignore_index=True,
                )

                results = (
                    results
                    .drop_duplicates(
                        subset=["id", "condition"],
                        keep="last",
                    )
                    .sort_values(["condition", "id"])
                    .reset_index(drop=True)
                )

                atomic_save_csv(results, output_path)

            acc = results.loc[
                results["condition"] == condition,
                "correct",
            ].mean()

            print(f"{condition} accuracy: {acc:.4f}")

    finally:
        del loaded
        gc.collect()
        torch.cuda.empty_cache()

    print("\nSaved:", output_path)
    return results


## 9. Optional smoke test

In [ ]:
# Recommended before full exploratory runs:
#
# smoke = evaluate_model(
#     "qwen2.5-3b",
#     limit_per_condition=5,
#     suffix="_smoke",
# )
#
# display(smoke[["id", "condition", "gold", "prediction", "correct"]])


# Primary model runs

These are intentionally commented because the three primary models have already been evaluated. If the full result CSVs remain in the Drive results folder, there is no need to rerun them.


In [ ]:
# qwen7b_results = evaluate_model("qwen2.5-7b")

In [ ]:
# mistral7b_results = evaluate_model("mistral-7b")

In [ ]:
# llama8b_results = evaluate_model("llama3.1-8b")

# Exploratory model runs

Run these one at a time.


In [14]:
qwen3b_results = evaluate_model("qwen2.5-3b")

Loading: Qwen/Qwen2.5-3B-Instruct
Candidate tokenization:
' A' -> [362]
' B' -> [425]
' C' -> [356]
' D' -> [422]
Scoring mode: single_token_fast


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


qwen2.5-3b | EN
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

EN accuracy: 0.8760

qwen2.5-3b | EN_MK
Pending: 895


...

EN_MK accuracy: 0.8279

qwen2.5-3b | MK
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK accuracy: 0.6380

qwen2.5-3b | MK_EN
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK_EN accuracy: 0.6447

Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/qwen2.5-3b_8bit_predictions.csv


In [15]:
llama3b_results = evaluate_model("llama3.2-3b")

Loading: meta-llama/Llama-3.2-3B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

EN_MK accuracy: 0.7721

llama3.2-3b | MK
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK accuracy: 0.5587

llama3.2-3b | MK_EN
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK_EN accuracy: 0.5743

Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/llama3.2-3b_8bit_predictions.csv


In [22]:
allam7b_results = evaluate_model("allam-7b")

Loading: humain-ai/ALLaM-7B-Instruct-preview
Revision: a895c826737f20b4e1caedbf5fbf6f1cb6c88b8a
Candidate tokenization:
' A' -> [395]
' B' -> [482]
' C' -> [415]
' D' -> [526]
Scoring mode: single_token_fast


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

EN_MK accuracy: 0.8335

allam-7b | MK
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK accuracy: 0.5508

allam-7b | MK_EN
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK_EN accuracy: 0.5296

Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/allam-7b_8bit_predictions.csv


## 10. Combine all currently available full-run result files

In [27]:
available_files = {}

for key in MODEL_SPECS:
    path = prediction_path_for(key)

    if path.exists():
        available_files[key] = path

print("Available full-run files:")
for key, path in available_files.items():
    print(key, "->", path.name)

if not available_files:
    raise FileNotFoundError("No full-run prediction files found.")

all_predictions = pd.concat(
    [pd.read_csv(path) for path in available_files.values()],
    ignore_index=True,
)

combined_available_path = (
    RESULTS_DIR
    / f"all_available_models_{QUANTIZATION}_predictions.csv"
)

all_predictions.to_csv(
    combined_available_path,
    index=False,
)

print("Combined rows:", len(all_predictions))
print("Saved:", combined_available_path)


Available full-run files:
qwen2.5-7b -> qwen2.5-7b_8bit_predictions.csv
mistral-7b -> mistral-7b_8bit_predictions.csv
llama3.1-8b -> llama3.1-8b_8bit_predictions.csv
qwen2.5-3b -> qwen2.5-3b_8bit_predictions.csv
llama3.2-3b -> llama3.2-3b_8bit_predictions.csv
allam-7b -> allam-7b_8bit_predictions.csv
Combined rows: 21480
Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/all_available_models_8bit_predictions.csv


## 11. Accuracy table for all currently available models

In [28]:
accuracy = (
    all_predictions
    .groupby(["model_key", "condition"], as_index=False)["correct"]
    .mean()
    .rename(columns={"correct": "accuracy"})
)

CONDITION_ORDER = ["EN", "EN_MK", "MK", "MK_EN"]

MODEL_ORDER = [
    "qwen2.5-7b",
    "llama3.1-8b",
    "mistral-7b",
    "qwen2.5-3b",
    "llama3.2-3b",
    "allam-7b",
]

accuracy_table = (
    accuracy
    .pivot(
        index="model_key",
        columns="condition",
        values="accuracy",
    )
    .reindex(columns=CONDITION_ORDER)
)

accuracy_table["delta_EN_to_EN_MK"] = (
    accuracy_table["EN_MK"] - accuracy_table["EN"]
)

accuracy_table["delta_MK_to_MK_EN"] = (
    accuracy_table["MK_EN"] - accuracy_table["MK"]
)

accuracy_table = accuracy_table.reindex(
    [m for m in MODEL_ORDER if m in accuracy_table.index]
)

summary_path = (
    RESULTS_DIR
    / f"accuracy_summary_available_models_{QUANTIZATION}.csv"
)

accuracy_table.to_csv(summary_path)

display((accuracy_table * 100).round(2))
print("Saved:", summary_path)


condition,EN,EN_MK,MK,MK_EN,delta_EN_to_EN_MK,delta_MK_to_MK_EN
model_key,,,,,,
qwen2.5-7b,90.73,88.38,73.63,72.85,-2.35,-0.78
llama3.1-8b,89.72,87.04,78.99,78.21,-2.68,-0.78
mistral-7b,81.79,78.32,58.99,57.65,-3.46,-1.34
qwen2.5-3b,87.60,82.79,63.80,64.47,-4.80,0.67
llama3.2-3b,84.25,77.21,55.87,57.43,-7.04,1.56
allam-7b,86.82,83.35,55.08,52.96,-3.46,-2.12


Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/accuracy_summary_available_models_8bit.csv


## 12. Final six-model integrity check

In [29]:
full_files = {
    key: prediction_path_for(key)
    for key in MODEL_SPECS
}

missing = [
    str(path)
    for path in full_files.values()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Six-model evaluation is incomplete. Missing:\n"
        + "\n".join(missing)
    )

all_six_predictions = pd.concat(
    [pd.read_csv(path) for path in full_files.values()],
    ignore_index=True,
)

expected_rows = 895 * 4 * 6

print("Expected rows:", expected_rows)
print("Found rows:", len(all_six_predictions))

assert len(all_six_predictions) == expected_rows

counts = (
    all_six_predictions
    .groupby(["model_key", "condition"])
    .size()
    .unstack()
)

display(counts)
assert (counts == 895).all().all()

expected_ids = set(df["id"].astype(int))

for model_key in MODEL_SPECS:
    for condition in CONDITIONS:
        subset = all_six_predictions[
            (all_six_predictions["model_key"] == model_key)
            & (all_six_predictions["condition"] == condition)
        ]

        assert set(subset["id"].astype(int)) == expected_ids

final_combined_path = (
    RESULTS_DIR
    / f"all_6_models_{QUANTIZATION}_predictions.csv"
)

all_six_predictions.to_csv(final_combined_path, index=False)

print("Final six-model integrity checks: PASS")
print("Saved:", final_combined_path)


Expected rows: 21480
Found rows: 21480


condition,EN,EN_MK,MK,MK_EN
model_key,,,,
allam-7b,895,895,895,895
llama3.1-8b,895,895,895,895
llama3.2-3b,895,895,895,895
mistral-7b,895,895,895,895
qwen2.5-3b,895,895,895,895
qwen2.5-7b,895,895,895,895


Final six-model integrity checks: PASS
Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/all_6_models_8bit_predictions.csv


## 13. Final six-model accuracy table

In [30]:
six_accuracy = (
    all_six_predictions
    .groupby(["model_key", "condition"], as_index=False)["correct"]
    .mean()
    .rename(columns={"correct": "accuracy"})
)

six_accuracy_table = (
    six_accuracy
    .pivot(
        index="model_key",
        columns="condition",
        values="accuracy",
    )
    .reindex(columns=CONDITION_ORDER)
    .reindex(MODEL_ORDER)
)

six_accuracy_table["delta_EN_to_EN_MK"] = (
    six_accuracy_table["EN_MK"] - six_accuracy_table["EN"]
)

six_accuracy_table["delta_MK_to_MK_EN"] = (
    six_accuracy_table["MK_EN"] - six_accuracy_table["MK"]
)

six_summary_path = (
    RESULTS_DIR
    / f"accuracy_summary_6_models_{QUANTIZATION}.csv"
)

six_accuracy_table.to_csv(six_summary_path)

display((six_accuracy_table * 100).round(2))
print("Saved:", six_summary_path)


condition,EN,EN_MK,MK,MK_EN,delta_EN_to_EN_MK,delta_MK_to_MK_EN
model_key,,,,,,
qwen2.5-7b,90.73,88.38,73.63,72.85,-2.35,-0.78
llama3.1-8b,89.72,87.04,78.99,78.21,-2.68,-0.78
mistral-7b,81.79,78.32,58.99,57.65,-3.46,-1.34
qwen2.5-3b,87.60,82.79,63.80,64.47,-4.80,0.67
llama3.2-3b,84.25,77.21,55.87,57.43,-7.04,1.56
allam-7b,86.82,83.35,55.08,52.96,-3.46,-2.12


Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/accuracy_summary_6_models_8bit.csv


## 14. Primary vs. exploratory tables

In [31]:
primary_keys = list(PRIMARY_MODEL_SPECS.keys())
exploratory_keys = list(EXPLORATORY_MODEL_SPECS.keys())

primary_table = six_accuracy_table.loc[primary_keys].copy()
exploratory_table = six_accuracy_table.loc[exploratory_keys].copy()

print("PRIMARY")
display((primary_table * 100).round(2))

print("\nEXPLORATORY")
display((exploratory_table * 100).round(2))

primary_table.to_csv(
    RESULTS_DIR
    / f"accuracy_summary_primary_3_models_{QUANTIZATION}.csv"
)

exploratory_table.to_csv(
    RESULTS_DIR
    / f"accuracy_summary_exploratory_3_models_{QUANTIZATION}.csv"
)


PRIMARY


condition,EN,EN_MK,MK,MK_EN,delta_EN_to_EN_MK,delta_MK_to_MK_EN
model_key,,,,,,
qwen2.5-7b,90.73,88.38,73.63,72.85,-2.35,-0.78
mistral-7b,81.79,78.32,58.99,57.65,-3.46,-1.34
llama3.1-8b,89.72,87.04,78.99,78.21,-2.68,-0.78



EXPLORATORY


condition,EN,EN_MK,MK,MK_EN,delta_EN_to_EN_MK,delta_MK_to_MK_EN
model_key,,,,,,
qwen2.5-3b,87.60,82.79,63.80,64.47,-4.80,0.67
llama3.2-3b,84.25,77.21,55.87,57.43,-7.04,1.56
allam-7b,86.82,83.35,55.08,52.96,-3.46,-2.12


## 15. Paired transition counts

In [32]:
def paired_transition_counts(
    predictions,
    model_key,
    baseline_condition,
    csw_condition,
):
    base = predictions[
        (predictions["model_key"] == model_key)
        & (predictions["condition"] == baseline_condition)
    ][["id", "correct"]].rename(
        columns={"correct": "base_correct"}
    )

    csw = predictions[
        (predictions["model_key"] == model_key)
        & (predictions["condition"] == csw_condition)
    ][["id", "correct"]].rename(
        columns={"correct": "csw_correct"}
    )

    merged = base.merge(csw, on="id", how="inner")
    assert len(merged) == 895

    return {
        "correct_to_correct": int(
            ((merged.base_correct == 1) & (merged.csw_correct == 1)).sum()
        ),
        "correct_to_wrong": int(
            ((merged.base_correct == 1) & (merged.csw_correct == 0)).sum()
        ),
        "wrong_to_correct": int(
            ((merged.base_correct == 0) & (merged.csw_correct == 1)).sum()
        ),
        "wrong_to_wrong": int(
            ((merged.base_correct == 0) & (merged.csw_correct == 0)).sum()
        ),
    }


transition_rows = []

for model_key in MODEL_SPECS:
    for base_cond, csw_cond in [
        ("EN", "EN_MK"),
        ("MK", "MK_EN"),
    ]:
        transition_rows.append({
            "model_key": model_key,
            "comparison": f"{base_cond}->{csw_cond}",
            **paired_transition_counts(
                all_six_predictions,
                model_key,
                base_cond,
                csw_cond,
            ),
        })

transition_table = pd.DataFrame(transition_rows)

transition_path = (
    RESULTS_DIR
    / "paired_transition_counts_6_models.csv"
)

transition_table.to_csv(transition_path, index=False)

display(transition_table)
print("Saved:", transition_path)


,model_key,comparison,correct_to_correct,correct_to_wrong,wrong_to_correct,wrong_to_wrong
0,qwen2.5-7b,EN->EN_MK,770,42,21,62
1,qwen2.5-7b,MK->MK_EN,608,51,44,192
2,mistral-7b,EN->EN_MK,670,62,31,132
3,mistral-7b,MK->MK_EN,452,76,64,303
4,llama3.1-8b,EN->EN_MK,762,41,17,75
5,llama3.1-8b,MK->MK_EN,664,43,36,152
6,qwen2.5-3b,EN->EN_MK,718,66,23,88
7,qwen2.5-3b,MK->MK_EN,496,75,81,243
8,llama3.2-3b,EN->EN_MK,668,86,23,118
9,llama3.2-3b,MK->MK_EN,438,62,76,319


Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/paired_transition_counts_6_models.csv


## 16. Exact McNemar tests

In [33]:
from scipy.stats import binomtest

mcnemar_rows = []

for _, row in transition_table.iterrows():
    b = int(row["correct_to_wrong"])
    c = int(row["wrong_to_correct"])
    n = b + c

    if n == 0:
        p_value = 1.0
    else:
        p_value = binomtest(
            k=min(b, c),
            n=n,
            p=0.5,
            alternative="two-sided",
        ).pvalue

    mcnemar_rows.append({
        "model_key": row["model_key"],
        "comparison": row["comparison"],
        "correct_to_wrong": b,
        "wrong_to_correct": c,
        "discordant_total": n,
        "exact_mcnemar_p": p_value,
    })

mcnemar_table = pd.DataFrame(mcnemar_rows)

mcnemar_path = RESULTS_DIR / "mcnemar_exact_6_models.csv"
mcnemar_table.to_csv(mcnemar_path, index=False)

display(mcnemar_table)
print("Saved:", mcnemar_path)


,model_key,comparison,correct_to_wrong,wrong_to_correct,discordant_total,exact_mcnemar_p
0,qwen2.5-7b,EN->EN_MK,42,21,63,1.114141e-02
1,qwen2.5-7b,MK->MK_EN,51,44,95,5.383940e-01
2,mistral-7b,EN->EN_MK,62,31,93,1.712764e-03
3,mistral-7b,MK->MK_EN,76,64,140,3.525838e-01
4,llama3.1-8b,EN->EN_MK,41,17,58,2.232552e-03
5,llama3.1-8b,MK->MK_EN,43,36,79,4.998969e-01
6,qwen2.5-3b,EN->EN_MK,66,23,89,5.690249e-06
7,qwen2.5-3b,MK->MK_EN,75,81,156,6.890543e-01
8,llama3.2-3b,EN->EN_MK,86,23,109,9.593063e-10
9,llama3.2-3b,MK->MK_EN,62,76,138,2.683900e-01


Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/mcnemar_exact_6_models.csv
